# 03 — Train the Multi-Task Model
Train `RadarMTL` to predict signal type + the four pulse parameters from raw I/Q.

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt

from radar.data import load_radchar, make_split, regression_targets, MinMaxNormaliser
from radar.model import RadarMTL

DATA_PATH = "../data/RadChar-Baseline.h5"
SEED = 42

# fix the seed so the split and training are reproducible
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# load raw IQ and the full label table
iq, labels = load_radchar(DATA_PATH)

In [ ]:
# split complex IQ into two channels: I (real) and Q (imaginary)
# (N, 512) complex -> (N, 2, 512) float
I = iq.real.astype(np.float32)
Q = iq.imag.astype(np.float32)
X = np.stack([I, Q], axis=1)  # (N, 2, 512)

# classification target: signal type (0..4)
y_cls = labels["signal_type"].astype(np.int64)

# regression targets: the 4 pulse parameters in real units (count + µs),
# ordered by radar.data.REGRESSION_TASKS = (n_pulses, pw, pri, td)
reg = regression_targets(labels)

# free the big complex array, we only need X now (it's ~4 GB)
del iq, I, Q

print("Input shape:", X.shape)
print("Class labels:", y_cls.shape, "| Regression targets:", reg.shape)

In [ ]:
# 70/15/15 split. save the indices so notebook 04 scores on the same test set
N = len(X)
train_idx, val_idx, test_idx = make_split(N, seed=SEED)

os.makedirs("../results", exist_ok=True)
np.save("../results/train_idx.npy", train_idx)
np.save("../results/test_idx.npy",  test_idx)

# standardise I/Q to the training mean/std (paper §3.1), fit on train rows only.
# save it so notebook 04 applies the same transform.
iq_mean = float(X[train_idx].mean())
iq_std  = float(X[train_idx].std())
X = ((X - iq_mean) / iq_std).astype(np.float32)
with open("../results/iq_stats.json", "w") as f:
    json.dump({"mean": iq_mean, "std": iq_std}, f, indent=2)

# scale the regression targets to [0, 1] using train min/max only (no leakage).
# save it so notebook 04 can convert predictions back to real units.
norm = MinMaxNormaliser.fit(reg[train_idx])
norm.save("../results/norm_stats.json")
reg_norm = norm.transform(reg).astype(np.float32)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
print("Saved split indices, iq_stats.json and norm_stats.json to results/")

In [ ]:
# dataset: returns (input, class label, 4 normalised regression targets)
class RadCharDataset(Dataset):
    def __init__(self, X, y_cls, y_reg):
        self.X = torch.tensor(X)
        self.y_cls = torch.tensor(y_cls)
        self.y_reg = torch.tensor(y_reg)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_cls[idx], self.y_reg[idx]


dataset = RadCharDataset(X, y_cls, reg_norm)
train_set = Subset(dataset, train_idx)
val_set   = Subset(dataset, val_idx)

# seed the shuffling so batch order is reproducible
g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, generator=g)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False)

In [ ]:
model = RadarMTL().to(device)

# loss = cross-entropy (signal type) + L1 on each regression task, weighted sum.
# weights follow the paper; change this dict to retune.
TASK_WEIGHTS = {"signal_type": 0.1, "n_pulses": 0.225, "pw": 0.225, "pri": 0.225, "td": 0.225}
REG_TASKS = RadarMTL.REGRESSION_TASKS  # (n_pulses, pw, pri, td)

ce = nn.CrossEntropyLoss()
l1 = nn.L1Loss()

def compute_losses(out, y_cls, y_reg):
    # per-task losses + the weighted total (y_reg columns follow REG_TASKS)
    losses = {"signal_type": ce(out["signal_type"], y_cls)}
    for i, task in enumerate(REG_TASKS):
        losses[task] = l1(out[task], y_reg[:, i])
    total = sum(TASK_WEIGHTS[t] * losses[t] for t in TASK_WEIGHTS)
    return total, losses

optimiser = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
print(model)

In [ ]:
# training loop with early stopping on the weighted val loss.
# also track val accuracy and per-task MAE in real units.
EPOCHS   = 100
PATIENCE = 10

# scale to turn normalised MAE back into real units
reg_scale = torch.tensor(norm.hi - norm.lo, dtype=torch.float32, device=device)

train_losses, val_losses, val_accs = [], [], []

best_val_loss    = float("inf")
patience_counter = 0
best_model_state = None

for epoch in range(EPOCHS):

    # --- train ---
    model.train()
    running = 0
    for X_b, yc_b, yr_b in train_loader:
        X_b, yc_b, yr_b = X_b.to(device), yc_b.to(device), yr_b.to(device)
        optimiser.zero_grad()
        out = model(X_b)
        loss, _ = compute_losses(out, yc_b, yr_b)
        loss.backward()
        optimiser.step()
        running += loss.item()
    train_loss = running / len(train_loader)

    # --- validate ---
    model.eval()
    val_loss = 0
    correct = total = 0
    abs_err = torch.zeros(len(REG_TASKS), device=device)  # summed |pred-target|, normalised
    with torch.no_grad():
        for X_b, yc_b, yr_b in val_loader:
            X_b, yc_b, yr_b = X_b.to(device), yc_b.to(device), yr_b.to(device)
            out = model(X_b)
            loss, _ = compute_losses(out, yc_b, yr_b)
            val_loss += loss.item()
            correct  += (out["signal_type"].argmax(1) == yc_b).sum().item()
            total    += yc_b.size(0)
            for i, task in enumerate(REG_TASKS):
                abs_err[i] += (out[task] - yr_b[:, i]).abs().sum()
    val_loss /= len(val_loader)
    val_acc = correct / total
    # normalised MAE -> real units
    val_mae = (abs_err / total * reg_scale).cpu().numpy()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # --- early stopping check ---
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        tag = " <- best"
    else:
        patience_counter += 1
        tag = f" (no improvement {patience_counter}/{PATIENCE})"

    mae_str = " ".join(f"{t}:{m:.2f}" for t, m in zip(REG_TASKS, val_mae))
    print(f"Epoch {epoch+1:03d}/{EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f} | acc {val_acc:.3f} | MAE {mae_str}{tag}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} — val loss stopped improving.")
        break

model.load_state_dict(best_model_state)
print(f"\nRestored best model (val loss: {best_val_loss:.4f})")

In [ ]:
# save weights for notebook 04
torch.save(model.state_dict(), "../results/radar_mtl.pth")
print("Model saved to results/radar_mtl.pth")

In [ ]:
# training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label="train loss")
axes[0].plot(val_losses,   label="val loss")
axes[0].set_title("Weighted total loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(val_accs, color="green", label="val signal-type accuracy")
axes[1].set_title("Validation classification accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()